# Working with pandas together with duckdb

## Loading sakila database from sqlite into duckdb

In [ ]:
import duckdb
from pathlib import Path

duckdb_path = "data/sakila.duckdb"

# deletes the path if it already exists
Path("data/sakila.duckdb").unlink(missing_ok=True)

with (
    duckdb.connect("data/sakila.duckdb") as conn,
    open("sql/load_sakila.sql") as ingest_script,
):
    conn.sql(ingest_script.read())
    
    films = conn.sql("FROM film;").df()

films.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,<NA>,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:00
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,<NA>,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2021-03-06 15:52:00
2,3,ADAPTATION HOLES,A Astounding Reflection of a Lumberjack And a ...,2006,1,<NA>,7,2.99,50,18.99,NC-17,"Trailers,Deleted Scenes",2021-03-06 15:52:00
3,4,AFFAIR PREJUDICE,A Fanciful Documentary of a Frisbee And a Lumb...,2006,1,<NA>,5,2.99,117,26.99,G,"Commentaries,Behind the Scenes",2021-03-06 15:52:00
4,5,AFRICAN EGG,A Fast-Paced Documentary of a Pastry Chef And ...,2006,1,<NA>,6,2.99,130,22.99,G,Deleted Scenes,2021-03-06 15:52:00


## Read in all data into dictionary of pandas dataframes

- nice data structure to work with 

In [ ]:
with duckdb.connect("data/sakila.duckdb") as conn:
    views_df = conn.sql(
        "FROM information_schema.views WHERE table_catalog = 'sakila';"
    ).df()

views_df


,table_catalog,table_schema,table_name,view_definition,check_option,is_updatable,is_insertable_into,is_trigger_updatable,is_trigger_deletable,is_trigger_insertable_into
0,sakila,main,actor,,NONE,NO,NO,NO,NO,NO
1,sakila,main,address,,NONE,NO,NO,NO,NO,NO
2,sakila,main,category,,NONE,NO,NO,NO,NO,NO
3,sakila,main,city,,NONE,NO,NO,NO,NO,NO
4,sakila,main,country,,NONE,NO,NO,NO,NO,NO
5,sakila,main,customer,,NONE,NO,NO,NO,NO,NO
6,sakila,main,customer_list,CREATE VIEW customer_list AS SELECT cu.custome...,NONE,NO,NO,NO,NO,NO
7,sakila,main,film,,NONE,NO,NO,NO,NO,NO
8,sakila,main,film_actor,,NONE,NO,NO,NO,NO,NO
9,sakila,main,film_category,,NONE,NO,NO,NO,NO,NO


In [26]:
views = {}
with duckdb.connect("data/sakila.duckdb") as conn:
    for name in views_df["table_name"]:
        views[name] = conn.sql(f"FROM {name};").df()

views.keys()

dict_keys(['actor', 'address', 'category', 'city', 'country', 'customer', 'customer_list', 'film', 'film_actor', 'film_category', 'film_list', 'film_text', 'inventory', 'language', 'payment', 'rental', 'sales_by_film_category', 'sales_by_store', 'staff', 'staff_list', 'store'])

In [29]:
views["film_actor"].head()

,actor_id,film_id,last_update
0,1,1,2021-03-06 15:52:45
1,1,23,2021-03-06 15:52:45
2,1,25,2021-03-06 15:52:45
3,1,106,2021-03-06 15:52:45
4,1,140,2021-03-06 15:52:45


## Check some film related dataframes

In [50]:
views["film"].head(2)

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
0,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,1,<NA>,6,0.99,86,20.99,PG,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:00
1,2,ACE GOLDFINGER,A Astounding Epistle of a Database Administrat...,2006,1,<NA>,3,4.99,48,12.99,G,"Trailers,Deleted Scenes",2021-03-06 15:52:00


In [48]:
views["film_category"].head(2)

,film_id,category_id,last_update
0,1,6,2021-03-06 15:53:28
1,2,11,2021-03-06 15:53:28


In [51]:
views["film_actor"].head(2)

,actor_id,film_id,last_update
0,1,1,2021-03-06 15:52:45
1,1,23,2021-03-06 15:52:45


In [52]:
views["actor"].head(2)

,actor_id,first_name,last_name,last_update
0,1.0,PENELOPE,GUINESS,2021-03-06 15:51:59
1,2.0,NICK,WAHLBERG,2021-03-06 15:51:59


## Join film related dataframes using duckdb

Joins are much cleaner to write in SQL than in pandas so we use duckdb to do the joins. To do that
we need to first register the dataframes as views in duckdb.

- duckdb parser looks for variable name and can't evaluate dictionary lookup so we need to register it into a view first

After joining, pick out the columns we want

In [82]:
film_names = ("film", "film_actor", "film_category", "actor", "category")

for film_name in film_names:
    duckdb.register(film_name, views[film_name])

films_joined = duckdb.sql("""
           SELECT 
                f.title, f.description, f.release_year, f.rental_duration, f.rating, 
                a.actor_id::INT AS actor_id, a.first_name AS actor_first_name, a.last_name AS actor_last_name, 
                c.name AS category 
           FROM film f
            LEFT JOIN film_actor fa ON f.film_id = fa.film_id 
            LEFT JOIN actor a ON a.actor_id = fa.actor_id
            LEFT JOIN film_category fc ON fc.film_id = f.film_id 
            LEFT JOIN category c ON c.category_id = fc.category_id 
           ;           
           """).df()

films_joined.columns

Index(['title', 'description', 'release_year', 'rental_duration', 'rating',
       'actor_id', 'actor_first_name', 'actor_last_name', 'category'],
      dtype='object')

In [83]:
films_joined.head()

,title,description,release_year,rental_duration,rating,actor_id,actor_first_name,actor_last_name,category
0,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,6,PG,1,PENELOPE,GUINESS,Documentary
1,ANACONDA CONFESSIONS,A Lacklusture Display of a Dentist And a Denti...,2006,3,R,1,PENELOPE,GUINESS,Animation
2,ANGELS LIFE,A Thoughtful Display of a Woman And a Astronau...,2006,3,G,1,PENELOPE,GUINESS,New
3,BULWORTH COMMANDMENTS,A Amazing Display of a Mad Cow And a Pioneer w...,2006,4,G,1,PENELOPE,GUINESS,Games
4,CHEAPER CLYDE,A Emotional Character Study of a Pioneer And a...,2006,6,G,1,PENELOPE,GUINESS,Sci-Fi


## Now do some EDA

We'll combine pandas and duckdb for this.

In [84]:
films_joined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5465 entries, 0 to 5464
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   title             5465 non-null   object
 1   description       5465 non-null   object
 2   release_year      5465 non-null   object
 3   rental_duration   5465 non-null   int64 
 4   rating            5465 non-null   object
 5   actor_id          5462 non-null   Int32 
 6   actor_first_name  5462 non-null   object
 7   actor_last_name   5462 non-null   object
 8   category          5465 non-null   object
dtypes: Int32(1), int64(1), object(7)
memory usage: 368.4+ KB


In [75]:
films_joined["rating"].value_counts()

rating
PG-13    1184
PG       1143
NC-17    1128
R        1033
G         977
Name: count, dtype: int64

In [87]:
# days?
films_joined["rental_duration"].value_counts()

rental_duration
6    1201
4    1113
3    1087
5    1057
7    1007
Name: count, dtype: int64

which top 10 actors have played in most films?

In [ ]:
# alternative is to use CTE 
actor_id_films = (
    duckdb.sql("""
    SELECT actor_id, COUNT(*) AS number_films 
    FROM films_joined
    GROUP BY actor_id
    ORDER BY number_films DESC;
""")
    .df()
)

actor_id_films.head()

,actor_id,number_films
0,107,42
1,102,41
2,198,40
3,181,39
4,23,37


In [118]:
duckdb.sql("""
    SELECT 
        f.actor_first_name,
        f.actor_last_name,
        a.number_films   
    FROM actor_id_films a
    LEFT JOIN films_joined f ON a.actor_id = f.actor_id
    GROUP BY ALL
    ORDER BY number_films DESC
""").df()

,actor_first_name,actor_last_name,number_films
0,GINA,DEGENERES,42
1,WALTER,TORN,41
2,MARY,KEITEL,40
3,MATTHEW,CARREY,39
4,SANDRA,KILMER,37
...,...,...,...
196,JULIA,ZELLWEGER,16
197,JULIA,FAWCETT,15
198,JUDY,DEAN,15
199,EMILY,DEE,14
